# 繪師 (Huishi) — Colab BackendGateway 啟動器

已自動設定白名單與 Google Drive 路徑：
- 白名單信箱：`qseft951753@gmail.com`
- 支援自動搜尋 `麻糬紅豆湯/繪師` 或任何掛載路徑下的 `AI_Studio`

In [ ]:
# @title 1. 掛載 Google Drive 與自動定位「繪師」目錄
import os, glob
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

candidates = [
    '/content/drive/MyDrive/麻糬紅豆湯/繪師',
    '/content/drive/MyDrive/繪師',
]
HUISHI_DIR = next((p for p in candidates if os.path.exists(os.path.join(p, 'backend_gateway.py'))), None)

if not HUISHI_DIR:
    hits = glob.glob('/content/drive/**/backend_gateway.py', recursive=True)
    if hits:
        HUISHI_DIR = os.path.dirname(hits[0])

if not HUISHI_DIR:
    raise FileNotFoundError('❌ 在 Google Drive 找不到 backend_gateway.py，請確認「麻糬紅豆湯/繪師」已加入「我的雲端硬碟」捷徑')

BASE_DIR = f'{HUISHI_DIR}/AI_Studio'
for sub in ['models', 'bgm', 'outputs', 'config']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

print('✅ 已定位繪師資料夾:', HUISHI_DIR)
print('✅ AI_Studio 目錄已就緒:', BASE_DIR)

In [ ]:
# @title 2. 安裝相依套件與 cloudflared
!pip install -q fastapi uvicorn firebase-admin pydantic pillow
!apt-get -qq update && apt-get -qq install -y ffmpeg
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print('✅ 套件、FFmpeg 與 cloudflared 安裝完成')

In [ ]:
# @title 3. 設定白名單與環境變數並載入 BackendGateway
import os, sys, glob, shutil

candidates = [
    '/content/drive/MyDrive/麻糬紅豆湯/繪師',
    '/content/drive/MyDrive/繪師',
]
HUISHI_DIR = next((p for p in candidates if os.path.exists(os.path.join(p, 'backend_gateway.py'))), None)
if not HUISHI_DIR:
    hits = glob.glob('/content/drive/**/backend_gateway.py', recursive=True)
    if hits:
        HUISHI_DIR = os.path.dirname(hits[0])

if not HUISHI_DIR:
    raise FileNotFoundError('❌ 找不到 backend_gateway.py，請先執行第 1 格並確認 Drive 已掛載')

BASE_DIR = f'{HUISHI_DIR}/AI_Studio'
os.environ['HUISHI_ALLOWED_EMAIL'] = 'qseft951753@gmail.com'
os.environ['FIREBASE_SERVICE_ACCOUNT_JSON'] = f'{BASE_DIR}/config/serviceAccountKey.json'
os.environ['HUISHI_OUTPUT_DIR'] = f'{BASE_DIR}/outputs'
os.environ['HUISHI_BGM_DIR'] = f'{BASE_DIR}/bgm'

# 複製至 /content 確保 Python import 穩定不受 Drive FUSE 延遲影響
for fname in ['backend_gateway.py', 'comfy_client.py', 'ffmpeg_composer.py']:
    src = os.path.join(HUISHI_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, f'/content/{fname}')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import backend_gateway
print('✅ 已載入 BackendGateway（來源:', HUISHI_DIR, '）')

In [ ]:
# @title 4. 啟動 FastAPI + Cloudflare Tunnel 並自動將最新 URL 寫入 Firestore system/gateway
import re
import subprocess
import threading
import time
from datetime import datetime
from zoneinfo import ZoneInfo
import uvicorn
from firebase_admin import firestore
import backend_gateway

backend_gateway.init_firebase_admin()

server_thread = threading.Thread(
    target=uvicorn.run,
    args=(backend_gateway.app,),
    kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'},
    daemon=True,
)
server_thread.start()
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if not tunnel_url:
    raise RuntimeError('❌ 無法取得 Cloudflare Tunnel URL')

gpu_info = backend_gateway.detect_gpu_vram()
db = firestore.client()
db.collection('system').document('gateway').set({
    'url': tunnel_url,
    'status': 'online',
    'updatedAt': datetime.now(ZoneInfo('Asia/Taipei')).isoformat(),
    'gpuName': gpu_info['gpu_name'],
    'vramGb': gpu_info['vram_gb'],
    'vramMode': gpu_info['vram_mode'],
    'workflowTier': gpu_info['workflow_tier'],
})

print('🚀 BackendGateway 已上線！')
print('🔗 Tunnel URL:', tunnel_url)
print('🖥️ GPU 資訊:', gpu_info)

tunnel_proc.wait()